# T3-05 · Heuristic first pass
Chạy sau khi notebook Prepare hoàn tất cell 3. Lượt này tạo nhãn nháp và danh sách uncertain; chưa phải gói bàn giao cho TV5.


In [ ]:
# @title 1. Mount Drive và lấy code T3-05 mới nhất
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys
REPO = Path('/content/arxiv-data-science')
BRANCH = 'feature/m3-tv3-auto-labeling'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', 'https://github.com/LeoTkiet/arxiv-data-science.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
sys.path.insert(0, str(REPO/'src'))
import arxiv_pipeline
print('Code:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
# @title 2. Gán nhãn heuristic lần đầu và thống kê uncertain
from arxiv_pipeline.matching.silver_first_pass import run_first_pass
BASE = Path('/content/drive/MyDrive/T3-05') # @param {type:'string'}
PREPARED = BASE/'output/prepared/prepared_manifest.json'
POPULATION = BASE/'population_manifest.json'
OUTPUT = BASE/'output/first_pass'
if not PREPARED.is_file():
    raise FileNotFoundError(f'Cell 3 của notebook Prepare chưa xuất xong: {PREPARED}')
summary = run_first_pass(PREPARED, POPULATION, OUTPUT)
print('Bài ready đã xử lý:', summary['prepared_ready'])
print('Bài phủ đủ tạm thời:', summary['candidate_full_coverage_publications'], '/', summary['minimum_auto_publications'])
print('Phủ đủ, có references:', summary['full_coverage_with_references'])
print('Phủ đủ, không có references:', summary['full_coverage_without_references'])
print('Phủ đủ và có matched:', summary['full_coverage_with_match'])
print('Dòng uncertain:', summary['uncertain_entries'])
print('Nhãn nháp:', OUTPUT/'draft_labels.json')
print('Danh sách uncertain:', OUTPUT/'uncertain_rows.json')
print('Báo cáo:', OUTPUT/'first_pass_manifest.json')
print('Bàn giao được:', summary['ready_for_handoff'])
